# Esercitazione: Classificazione Land Cover con Machine Learning

**Obiettivo:** Costruire un modello di classificazione supervisionata multiclasse (Acqua, Foresta, Suolo) trasformando dati spaziali in formato tabellare.

### **Sommario delle attività**
1. **Download STAC API**: Scaricare porzioni di Sentinel-2 per l'area di addestramento e di test.
2. **Data Preparation**: Estrarre pixel spaziali da GeoTIFF, incrociarli con le etichette fornite e convertire in tabular data (`pandas`).
3. **Machine Learning Supervisionato**: Addestrare `RandomForestClassifier` (o altro) e generare mappe predittive su nuovi dati.
4. **Machine Learning Non Supervisionato**: Confrontare i risultati con la segmentazione non supervisionata `KMeans`.
5. **Esportazione GIS**: Salvare la mappa finale in formato GeoTIFF compatibile con QGIS/ArcGIS.

## Setup iniziale
Installare le librerie richieste e importare i moduli principali.

In [ ]:
!pip install rasterio rioxarray pystac-client scikit-learn matplotlib pandas numpy xarray

In [ ]:
import os
import rasterio
import rioxarray
import xarray
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pystac_client import Client
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.cluster import KMeans

import warnings
warnings.filterwarnings('ignore')

os.environ['AWS_NO_SIGN_REQUEST'] = 'YES'
os.environ['GDAL_DISABLE_READDIR_ON_OPEN'] = 'EMPTY_DIR'

## Intro esercitazione
Qui ci sono i passi principali che completeremo insieme.

1. Scaricare due immagini Sentinel-2 via STAC.
2. Estrarre le bande Blu, Verde, Rosso, NIR e creare un GeoTIFF locale.
3. Addestrare Random Forest e valutare il modello.
4. Applicare KMeans per clustering.
5. Esportare mappa in GeoTIFF.

## 1) Download STAC e creazione TIFF
Compila la funzione di download e crea i file `Train_Brescia.tif` e `Test_Iseo.tif`.

In [ ]:
STAC_API_URL = 'https://earth-search.aws.element84.com/v1'
client = Client.open(STAC_API_URL)

PUNTI = {
    'Train_Brescia': {'lon': 10.27, 'lat': 45.50},
    'Test_Iseo': {'lon': 10.05, 'lat': 45.65}
}

BUFFER_DEG = 0.05
BANDS = ["blue", "green", "red", "nir"]
YEAR = '2023'

def download_point(name, lon, lat):
    file_out = f'{name}.tif'
    if os.path.exists(file_out):
        return file_out

    bbox = [lon - BUFFER_DEG, lat - BUFFER_DEG, lon + BUFFER_DEG, lat + BUFFER_DEG]

    # HINT: cerca l'item con nuvolosita minima e scarica le bande ["blue", "green", "red", "nir"]

    search = client.search(
        collections=["sentinel-2-l2a"],
        bbox=bbox,
        datetime=f"{YEAR}-06-01/{YEAR}-08-31",
        query={"eo:cloud_cover": {"lt": 10}}
    )
    items = list(search.items())
    best_item =

    band_arrays = []
    for band in BANDS:
        href = best_item.assets[band].href
        ds = rioxarray.open_rasterio(href)
        # HINT: usa ds.rio.clip_box(*bbox, crs='EPSG:4326')
        cropped =

    # HINT: unisci le bande con xarray.concat(...)
    merged =
    # HINT: scrivi il tif con merged.rio.to_raster(...)
    # ...
    return file_out

# scarica e salva Train_Brescia.tif e Test_Iseo.tif
train_img_path = download_point('Train_Brescia', PUNTI['Train_Brescia']['lon'], PUNTI['Train_Brescia']['lat'])
test_img_path = download_point('Test_Iseo', PUNTI['Test_Iseo']['lon'], PUNTI['Test_Iseo']['lat'])

Visualizziamo l'immagine

In [ ]:
# Leggiamo l'immagine di training
src_train = rasterio.open(train_img_path)
img_train = src_train.read()

print("Shape immagine (Bande, Altezza, Larghezza):", img_train.shape)

# Plot RGB (Bande 3, 2, 1 che corrispondono a R, G, B nel nostro cubo 4-bande)
# Rasterio ordina 1-based: [Blu, Verde, Rosso, NIR] -> [2, 1, 0] per RGB
rgb_img = np.dstack((img_train[2], img_train[1], img_train[0]))

# Normalizzazione per visualizzazione
rgb_img = rgb_img / np.percentile(rgb_img, 98)
rgb_img = np.clip(rgb_img, 0, 1)

plt.figure(figsize=(8,8))
plt.imshow(rgb_img)
plt.title("Immagine di Training (RGB)")
plt.axis('off')
plt.show()

## 2) Preparazione dataset e label
Adesso useremo il file `training_points.csv`.
Estrarremo i valori delle bande e creare un DataFrame con `['blue', 'green', 'red', 'nir',Label]` che farà da nostro dataset

In [ ]:
training_csv = 'training_points.csv'

# 0 > Acqua
# 1 > Foresta/Erba
# 2 > Urbano/Suolo non coltivato

# HINT: usa pandas.read_csv('training_points.csv')
df_truth = pd.read_csv(training_csv)

# HINT: andiamo a recuperare dall'immagine i valori corrispondendi ad ogni label
features = []
for index, row in df_truth.iterrows():
    # img_train ha forma (4, H, W)
    # ...
    features.append()

# HINT: crea DataFrame features_df con colonne ['blue', 'green', 'red', 'nir'],
# con i valori dei punti dell'immagine scaricata
features_df =

# HINT: concatena features_df con df['Class']
dataset =


## 3) Addestramento supervisionato
Addestriamo RandomForest usando le feature estratte.

In [ ]:
# HINT: X =

# HINT: y =

# HINT: X_train, X_test, y_train, y_test = train_test_split(...)

# HINT: RandomForestClassifier o altro a preferenza

model = None

# addestrare modello e stampare classification_report






## 4) Inferenza spaziale
Trasformiamo intero GeoTIFF in matrice di pixel e generiamo la mappa usando il classificatore addestrato.

In [ ]:
def elabora_e_classifica(tif_path, model):
    with rasterio.open(tif_path) as src:
        img_data = src.read()
        # img_data ha forma (4, H, W)

        n_bands, h, w =

        # HINT: flat_img = img_data.reshape().T
        flat_img =

        # HINT: model.predict(flat_img) e reshape su (h,w)
        print(f"Predizione su {h*w} pixel per {tif_path}...")
        pred_flat =

        # Reshape per tornare alle due dimensioni: (H*W) -> (H, W)
        pred_map =
        return pred_map


map_train = elabora_e_classifica(train_img_path, model)
map_test = elabora_e_classifica(test_img_path, model)


In [ ]:
# Plot Risultati
from matplotlib.colors import ListedColormap

cmap = ListedColormap(['blue', 'forestgreen', 'saddlebrown'])

fig, axes = plt.subplots(1, 2, figsize=(15, 7))

# TIF 1: Train
axes[0].imshow(map_train, cmap=cmap)
axes[0].set_title('Predizione Brescia (Train Area)')
axes[0].axis('off')

# TIF 2: Test
axes[1].imshow(map_test, cmap=cmap)
axes[1].set_title('Predizione Iseo (Test Area)')
axes[1].axis('off')

plt.show()

## 5) Clustering non supervisionato
Qui applichiamo KMeans su `test_img_path` e confrontiamo con la predizione supervisata.

In [ ]:
from sklearn.cluster import KMeans

print("Esecuzione KMeans su area di Test...")

with rasterio.open(test_img_path) as src:
    test_data = src.read()
    h = test_data.shape[1]
    w = test_data.shape[2]

    # Flattening per KMeans
    flat_img =

# Crea modello kmeans
kmeans =

# HINT: fit_predict
clusters_flat =

# HINT: reshape
map_cluster =

# PLOT CONFRONTO
fig, axes = plt.subplots(1, 2, figsize=(15, 7))

axes[0].imshow(map_test, cmap=cmap)
axes[0].set_title('Supervisionato (Random Forest)')
axes[0].axis('off')

# I colori di KMeans sono arbitrari (non mappati a classi specifiche)
axes[1].imshow(map_cluster, cmap='viridis')
axes[1].set_title('Non Supervisionato (KMeans)')
axes[1].axis('off')
plt.show()


## 6) Esportazione GeoTIFF
Salviamo la mappa di clustering in un nuovo file `iseo_kmeans_clusters.tif`.

In [ ]:
with rasterio.open(test_img_path) as src:
    profile = src.profile

out_tif = 'iseo_kmeans_clusters.tif'
print(f"Salvataggio mappa clustering in {out_tif}...")

with rasterio.open(test_img_path) as src:
    profile = src.profile

    # Aggiorniamo info profilo (1 banda con la label, intero a 8 bit)
    profile.update(
        dtype=rasterio.uint8,
        count=1,
        compress='deflate'
    )

    # Scrittura nuovo file
    with rasterio.open(out_tif, 'w', **profile) as dst:
        dst.write(map_cluster.astype(rasterio.uint8), 1)

print("Esportazione completata!")